In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

PMF_TOTAL      = "../data/pmf_data/total"
DIST_TOTAL_DIR = "../data/distribution_data/total"    # summary_{analog}.dat
MACCALLUM_DIR  = "../../not_avail/maccallum"

REGION_BOUNDS   = [9.5, 19.5, 30.0]
MACCALLUM_ZERO  = 37.0                     # Å – MacCallum reference
OUR_COLOR       = 'tab:blue'
MAC_COLOR       = '#7f7f7f'                # muted grey (external data)

# ─── PMF loaders ────────────────────────────────────────────
def load_our_pmf(analog):
    path = os.path.join(PMF_TOTAL, analog, f"pmf_{analog}.dat")
    if not os.path.isfile(path):
        return None, None, None
    df = pd.read_csv(path, sep=r'\s+')
    return df['z'].to_numpy(), df['mean'].to_numpy(), df['se'].to_numpy()

def plateau_end(analog, z_max=20.0):
    """Return the upper z bound of the null-density plateau starting at z=0,
    looking up to z_max. Returns 0 if there is no plateau."""
    path = os.path.join(DIST_TOTAL_DIR, analog, f"summary_{analog}.dat")
    if not os.path.isfile(path):
        return 0.0
    df = pd.read_csv(path, sep=r'\s+')
    df = df[(df['z'] >= 0) & (df['z'] <= z_max)].sort_values('z').reset_index(drop=True)
    end = 0.0
    for _, row in df.iterrows():
        if row['mean'] == 0:
            end = row['z'] + 0.5
        else:
            break
    return end

def plot_pmf_with_plateau(ax, x, y, pe, color, lw, label=None, alpha=1.0):
    """Plot y vs x with a dashed segment on the null-density plateau (x ≤ pe)
    and a solid segment beyond it (x ≥ pe), sharing the same color. The
    legend label is attached only to the solid segment."""
    x = np.asarray(x)
    y = np.asarray(y)
    if pe > 0:
        mask_dash  = x <= pe
        mask_solid = x >= pe   # overlap at pe keeps the curve continuous
    else:
        mask_dash  = np.zeros_like(x, dtype=bool)
        mask_solid = np.ones_like(x, dtype=bool)
    if mask_solid.any():
        ax.plot(x[mask_solid], y[mask_solid], '-',
                color=color, lw=lw, alpha=alpha, label=label)
    if mask_dash.any():
        ax.plot(x[mask_dash], y[mask_dash], '--',
                color=color, lw=lw, alpha=alpha)

def zero_at(z, pmf, z_ref=MACCALLUM_ZERO):
    m = np.isfinite(z) & np.isfinite(pmf)
    if m.sum() == 0:
        return pmf
    order = np.argsort(z[m])
    return pmf - np.interp(z_ref, z[m][order], pmf[m][order])

def load_maccallum(filename):
    path = os.path.join(MACCALLUM_DIR, filename)
    if not os.path.isfile(path):
        return None, None, None
    arr = np.loadtxt(path)
    z   = arr[:, 0] * 10.0                 # nm → Å
    pmf = arr[:, 1]
    se  = arr[:, 2] if arr.shape[1] >= 3 else None
    pmf = zero_at(z, pmf)
    return z, pmf, se

# ─── Sub-plot helper ────────────────────────────────────────
def plot_analog(ax, analog, label, maccallum_file):
    plotted = False
    z, y, e = load_our_pmf(analog)
    if z is not None:
        pe = plateau_end(analog)
        plot_pmf_with_plateau(ax, z, y, pe,
                              color=OUR_COLOR, lw=1.5, label='This study')
        if e is not None:
            ax.fill_between(z, y - e, y + e, color=OUR_COLOR,
                            alpha=0.25, linewidth=0)
        plotted = True

    z_m, y_m, e_m = load_maccallum(maccallum_file)
    if z_m is not None:
        ax.plot(z_m, y_m, color=MAC_COLOR, lw=1.5, linestyle='-',
                label='MacCallum et al.')
        if e_m is not None:
            ax.fill_between(z_m, y_m - e_m, y_m + e_m,
                            color=MAC_COLOR, alpha=0.20, linewidth=0)
        plotted = True

    for xv in REGION_BOUNDS:
        ax.axvline(x=xv, linestyle='--', alpha=0.7, linewidth=0.8,
                   color='gray', zorder=0)

    ax.set_xlim(0, 35)
    ax.axhline(0.0, color='k', lw=0.5, alpha=0.5)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.grid(True, which='major', linestyle='--', alpha=0.4, linewidth=0.3)
    ax.tick_params(axis='both', labelsize=8)
    if not plotted:
        ax.text(0.5, 0.5, 'no data', transform=ax.transAxes,
                ha='center', va='center', color='gray')

# ─── Grid-plot helper ───────────────────────────────────────
def make_grid(entries, ncols, name):
    nrows = int(np.ceil(len(entries) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(12, 2.2 * nrows),
                              sharex=True, dpi=2000,
                              gridspec_kw={'hspace': 0.35, 'wspace': 0.25})
    axes_flat = np.atleast_2d(axes).ravel()

    for ax, (analog, label, mac_file) in zip(axes_flat, entries):
        plot_analog(ax, analog, label, mac_file)

    for ax in axes_flat[len(entries):]:
        ax.set_visible(False)

    # Legend once, taken from the first populated axes.
    for ax in axes_flat:
        h, l = ax.get_legend_handles_labels()
        if h:
            ax.legend(h, l, fontsize=10, frameon=False, loc='upper left')
            break

    fig.text(0.5, 0.02, r"z (Å)", ha='center', fontsize=11)
    fig.text(0.075, 0.5, "PMF (kJ/mol)", va='center',
             rotation='vertical', fontsize=11)

    os.makedirs("../plot", exist_ok=True)
    out = f"../plot/{name}.png"
    plt.tight_layout(rect=[0.04, 0.03, 1, 1])
    plt.savefig(out, dpi=600, bbox_inches='tight')
    plt.show()
    print("Saved →", out)

# ─── Groups ─────────────────────────────────────────────────
NEUTRAL_HYDROPHOBIC = [
    ('sca', 'ALA', 'ala.dat'),
    ('scv', 'VAL', 'val.dat'),
    ('scl', 'LEU', 'leu.dat'),
    ('sci', 'ILE', 'ile.dat'),
    ('scc', 'CYS', 'cys.dat'),
    ('scm', 'MET', 'met.dat'),
    ('scs', 'SER', 'ser.dat'),
    ('sct', 'THR', 'thr.dat'),
    ('scn', 'ASN', 'asn.dat'),
    ('scq', 'GLN', 'gln.dat'),
    ('scf', 'PHE', 'phe.dat'),
    ('scy', 'TYR', 'tyr.dat'),
    ('scw', 'TRP', 'trp.dat'),
]

TITRATABLE = [
    ('scd',  'ASP$^-$', 'asp-.dat'),
    ('scdn', 'ASP$^0$', 'asp0.dat'),
    ('sce',  'GLU$^-$', 'glu-.dat'),
    ('scen', 'GLU$^0$', 'glu0.dat'),
    ('sck',  'LYS$^+$', 'lys+.dat'),
    ('sckn', 'LYS$^0$', 'lys0.dat'),
    ('scr',  'ARG$^+$', 'arg+.dat'),
    ('scrn', 'ARG$^0$', 'arg0.dat'),
]

make_grid(NEUTRAL_HYDROPHOBIC, ncols=4, name='FigureS12')
make_grid(TITRATABLE,          ncols=4, name='FigureS13')
